# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SDlel/ml-assignments/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

My research question is: which observed content pages should be reviewed first for a possible refresh? The decision this supports is a content team's weekly prioritization queue, where reviewing the right pages first matters more than labeling every page perfectly.

In [1]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "scripts").exists())

WORK_OUTPUTS = ROOT / "work" / "outputs"
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import display_path


def run_script(script_name):
    subprocess.run([sys.executable, str(ROOT / "scripts" / script_name)], cwd=ROOT, check=True)

for script in ["01_prepare_features.py", "02_baseline_score.py", "03_train_model.py", "04_evaluate_and_export.py"]:
    run_script(script)

summary = json.loads((ROOT / "outputs" / "summary.json").read_text())
model_results = json.loads((ROOT / "outputs" / "model_results.json").read_text())
feature_meta = json.loads((ROOT / "data" / "processed" / "feature_metadata.json").read_text())
queue = pd.read_csv(ROOT / "outputs" / "refresh_queue.csv")

pd.DataFrame([
    {"item": "rows scored", "value": summary["rows_scored"]},
    {"item": "best model", "value": summary["best_model"]},
    {"item": "queue output", "value": summary["queue_output"]},
])

,item,value
0,rows scored,30000
1,best model,random_forest
2,queue output,outputs\refresh_queue.csv


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

I used the bundled anonymized FlyRank content refresh export in `data/raw/content_refresh_anonymized.csv`, which represents the March 2026 content performance data used for this repo. One row is one content page for one client on one reporting date. I excluded private identifiers like URLs, domains, titles, and keywords, and I did not use future information or product flags as model inputs.

In [2]:
data_summary = pd.DataFrame([
    {"field": "raw rows", "value": feature_meta["initial_rows"]},
    {"field": "prepared rows", "value": feature_meta["prepared_rows"]},
    {"field": "declining rows", "value": feature_meta["declining_rows"]},
    {"field": "declining rate", "value": round(feature_meta["declining_rate"], 3)},
    {"field": "unit", "value": "one content page for one client on one reporting date"},
])
data_summary.to_csv(WORK_OUTPUTS / "capstone_data_summary.csv", index=False)
data_summary

,field,value
0,raw rows,30000
1,prepared rows,30000
2,declining rows,16262
3,declining rate,0.542
4,unit,one content page for one client on one reporti...


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

The label is `trend_direction == down`, which is an observed decline proxy rather than a perfect truth about content quality. The baseline is a hand-written refresh score using observed visibility, freshness, position, and content depth signals. The models use numeric and categorical features available at prediction time and are validated with a client-holdout split.

In [3]:
methodology = pd.DataFrame([
    {"part": "label", "choice": model_results["target"]},
    {"part": "split", "choice": model_results["split_strategy"]},
    {"part": "baseline", "choice": "transparent rules score"},
    {"part": "models", "choice": ", ".join(model_results["models"].keys())},
    {"part": "feature count", "choice": model_results["feature_count"]},
])
methodology.to_csv(WORK_OUTPUTS / "capstone_methodology.csv", index=False)
methodology

,part,choice
0,label,is_declining_label
1,split,client_holdout
2,baseline,transparent rules score
3,models,"decision_tree, logistic_regression, random_forest"
4,feature count,52


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

On the measured client-holdout split, the random forest had the best Precision@50. This is a directional result for prioritizing manual review, not proof that the model knows which edits will improve rankings.

In [4]:
result_rows = []
for name, metrics in model_results["models"].items():
    result_rows.append({
        "method": name,
        "roc_auc": metrics["roc_auc"],
        "average_precision": metrics["average_precision"],
        "precision_at_50": metrics["precision_at_50"],
    })
baseline = model_results["baseline"]
result_rows.append({
    "method": "baseline_rules",
    "roc_auc": baseline["baseline_roc_auc"],
    "average_precision": baseline["baseline_average_precision"],
    "precision_at_50": baseline["baseline_precision_at_50"],
})
results_table = pd.DataFrame(result_rows).sort_values("precision_at_50", ascending=False)
results_table.to_csv(WORK_OUTPUTS / "capstone_results_vs_baseline.csv", index=False)
results_table.round(3)

,method,roc_auc,average_precision,precision_at_50
2,random_forest,0.747,0.610,0.68
0,decision_tree,0.742,0.575,0.62
1,logistic_regression,0.700,0.522,0.40
3,baseline_rules,0.627,0.468,0.24


## 5. Limitations

*What this work cannot claim.*

This work cannot claim that a refresh caused performance changes, that the queue is correct for every client, or that the model predicts a search engine algorithm. It uses an anonymized sample and an observed decline proxy, so the safest claim is decision support for manual prioritization.

In [5]:
limitations = pd.DataFrame({"limitation": [
    "observed decline is a proxy label, not ground truth content quality",
    "client-holdout validation is stronger than row-random validation but still offline",
    "the queue does not include live editorial context or SERP review",
    "private page text, URLs, and keywords were intentionally excluded",
]})
limitations.to_csv(WORK_OUTPUTS / "capstone_limitations.csv", index=False)
limitations

,limitation
0,"observed decline is a proxy label, not ground ..."
1,client-holdout validation is stronger than row...
2,the queue does not include live editorial cont...
3,"private page text, URLs, and keywords were int..."


## 6. Ranked recommendations

*The action playbook output ? the paper's recommendations section.*

The first recommendation is to review high-confidence pages with both measured demand and model decline risk. The second is to check CTR or engagement reason codes before editing, because those can have non-content explanations. The third is to treat low-confidence rows as monitoring candidates unless a human reviewer has extra context.

In [6]:
recommendations = queue.head(20)[[
    "final_rank", "content_id", "final_refresh_score", "confidence", "suggested_action",
    "final_reason_codes", "impressions_90d", "sessions_90d", "trend_direction",
]].copy()
recommendations.to_csv(WORK_OUTPUTS / "capstone_top_recommendations.csv", index=False)
recommendations.head(10)

,final_rank,content_id,final_refresh_score,confidence,suggested_action,final_reason_codes,impressions_90d,sessions_90d,trend_direction
0,1,content_1f080331fa2b,81.928467,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,66,down
1,2,content_6aa43079fb0c,81.728449,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,23,down
2,3,content_d6570c51c9bd,81.639118,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2498,9,down
3,4,content_e04eb9549989,80.804986,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3393,5,down
4,5,content_72e800a9c214,80.801530,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,27,down
5,6,content_9b6df29f7889,80.752578,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1622,20,down
6,7,content_b69288c5e701,80.632372,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,5811,14,down
7,8,content_ba6f9dfcbca1,80.439403,medium,refresh,declining_with_demand|model_decline_risk|visib...,4366,8,down
8,9,content_b1d593faf9c6,80.204325,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,2655,92,down
9,10,content_bb6ebb5ec8c8,80.146808,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2621,10,down


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The paper should embed the model comparison table, final queue preview, action mix, confidence mix, reason-code chart, feature-importance chart, and trend distribution chart. These are all generated from repository outputs.

In [7]:
charts_dir = WORK_OUTPUTS / "charts"
charts_dir.mkdir(parents=True, exist_ok=True)
for chart_path in (ROOT / "outputs" / "charts").glob("*.svg"):
    shutil.copy2(chart_path, charts_dir / chart_path.name)

artifact_index = pd.DataFrame([
    {"artifact": "capstone_results_vs_baseline.csv", "path": display_path(WORK_OUTPUTS / "capstone_results_vs_baseline.csv")},
    {"artifact": "capstone_top_recommendations.csv", "path": display_path(WORK_OUTPUTS / "capstone_top_recommendations.csv")},
    {"artifact": "action_mix.svg", "path": display_path(charts_dir / "action_mix.svg")},
    {"artifact": "confidence_mix.svg", "path": display_path(charts_dir / "confidence_mix.svg")},
    {"artifact": "top_reason_codes.svg", "path": display_path(charts_dir / "top_reason_codes.svg")},
    {"artifact": "top_feature_importance.svg", "path": display_path(charts_dir / "top_feature_importance.svg")},
    {"artifact": "trend_distribution.svg", "path": display_path(charts_dir / "trend_distribution.svg")},
])
artifact_index.to_csv(WORK_OUTPUTS / "capstone_artifact_index.csv", index=False)
artifact_index

,artifact,path
0,capstone_results_vs_baseline.csv,work\outputs\capstone_results_vs_baseline.csv
1,capstone_top_recommendations.csv,work\outputs\capstone_top_recommendations.csv
2,action_mix.svg,work\outputs\charts\action_mix.svg
3,confidence_mix.svg,work\outputs\charts\confidence_mix.svg
4,top_reason_codes.svg,work\outputs\charts\top_reason_codes.svg
5,top_feature_importance.svg,work\outputs\charts\top_feature_importance.svg
6,trend_distribution.svg,work\outputs\charts\trend_distribution.svg


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled ? markdown thinking AND code where code is requested.
- [x] Every number comes from a query, dataframe, or saved output.
- [x] I used careful language: observed / measured / directional / decision-support.
- [x] I did not include private client names, domains, URLs, titles, or keywords.
- [x] The capstone conclusions are cautious and trace back to generated outputs.